# 3. LangChain

[Docling Tutorial](https://docling-project.github.io/docling/examples/rag_langchain/#setup)

While the original tutorial use HF inference points, I have modified the script to use local Ollama. 

In [5]:
import os
from pathlib import Path
from tempfile import mkdtemp

from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_docling.loader import ExportType

load_dotenv()

# https://github.com/huggingface/transformers/issues/5486:
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
FILE_PATH = r"my_papers/2024 - 15.pdf"


**DoclingLoader**

It loads PDFs and converts them into a Document Document and aslo chunking is done by hybrid chunker. We get ready chunks for embedding. 

In [7]:
# Document Loading
from langchain_docling import DoclingLoader
from docling.chunking import HybridChunker

loader = DoclingLoader(
    file_path=FILE_PATH,
    chunker=HybridChunker(),
)

docs = loader.load()

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-02-11 19:30:58,795 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-11 19:30:58,833 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-02-11 19:30:58,836 [RapidOCR] main.py:53: Using C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-02-11 19:30:59,012 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-11 19:30:59,043 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-02-11 19:30:59,044 [Ra

In [8]:
# If Export Type DOC_CHUNKS
splits = docs

**Ingestion**

*If Using Milvus Database run following in WSL Terminal*

``pip install milvus`` (already installed)

To start a milvus server: ``milvus-server --proxy-port 19531``

**If using OllAMA for Embeddings**

In [10]:
OLLAMA_EMBED_MODEL = "mxbai-embed-large:latest"
OLLAMA_LLM_MODEL = "llama3.1:8b-instruct-q4_K_M"

In [20]:
from langchain_milvus import Milvus
from langchain_community.embeddings import OllamaEmbeddings

embedding = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL)

vectorstore = Milvus.from_documents(
    documents=splits,
    embedding=embedding,
    collection_name="docling_demo",
    # Connect to the WSL server via localhost
    connection_args={"uri": "http://localhost:19531"}, 
    drop_old=True,
)

**Using OpenAI Embeddings and Model**

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    # With the `text-embedding-3` class
    # of models, you can specify the size
    # of the embeddings you want returned.
    # dimensions=1024
)

In [9]:
from langchain_milvus import Milvus

vectorstore = Milvus.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="docling_demo",
    # Connect to the WSL server via localhost
    connection_args={"uri": "http://localhost:19531"}, 
    drop_old=True,
)

# 4. RAG

In [13]:
QUESTION = "Which are the main findings of this paper?"
PROMPT = PromptTemplate.from_template(
    "Context information is below.\n---------------------\n{context}\n---------------------\nGiven the context information and not prior knowledge, answer the query.\nQuery: {input}\nAnswer:\n",
)
TOP_K = 5


**Using Ollama LLM**

In [ ]:
OLLAMA_EMBED_MODEL = "mxbai-embed-large:latest"
OLLAMA_LLM_MODEL = "llama3.1:8b-instruct-q4_K_M"

In [27]:
#from langchain.chains import create_retrieval_chain
from langchain_classic.chains import create_retrieval_chain
#from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.chat_models import ChatOllama

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
llm = ChatOllama(
    model=OLLAMA_LLM_MODEL,
    temperature=0.5,
)


def clip_text(text, threshold=100):
    return f"{text[:threshold]}..." if len(text) > threshold else text

In [28]:
question_answer_chain = create_stuff_documents_chain(llm, PROMPT)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
resp_dict = rag_chain.invoke({"input": QUESTION})

clipped_answer = clip_text(resp_dict["answer"], threshold=200)
print(f"Question:\n{resp_dict['input']}\n\nAnswer:\n{clipped_answer}")
for i, doc in enumerate(resp_dict["context"]):
    print()
    print(f"Source {i + 1}:")
    print(f"  text: {json.dumps(clip_text(doc.page_content, threshold=350))}")
    for key in doc.metadata:
        if key != "pk":
            val = doc.metadata.get(key)
            clipped_val = clip_text(val) if isinstance(val, str) else val
            print(f"  {key}: {clipped_val}")

Question:
Which are the main findings of this paper?

Answer:
Based on the provided context, it appears that the paper presents a framework for merging sustainability metrics with Artificial Intelligence (AI). The main findings can be inferred from the following...

Source 1:
  text: "3. Research methods\nThe overall research framework of the paper is shown in Figure 2."
  source: my_papers/2024 - 15.pdf
  dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/71', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 5, 'bbox': {'l': 99.78, 't': 479.4980024414062, 'r': 345.705, 'b': 470.9380024414062, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 65]}]}], 'headings': ['3. Research methods'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 18434708872324074617, 'filename': '2024 - 15.pdf'}}

Source 2:
  text: "5.4 Research limitations and future prospect

**Using OpenAI LLM**

In [23]:
QUESTION = "Classify this paper into one of the following categories: descriptive, exploratory, inferential, predictive, causal. Also justify why you classified it into that category based on evidences from the paper."

In [11]:
from langchain_openai import ChatOpenAI

In [24]:

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

llm = ChatOpenAI(
    model="gpt-4o-mini", # or "gpt-4o"
    temperature=0.5,
)

def clip_text(text, threshold=10000):
    return f"{text[:threshold]}..." if len(text) > threshold else text

In [25]:
import json

question_answer_chain = create_stuff_documents_chain(llm, PROMPT)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
resp_dict = rag_chain.invoke({"input": QUESTION})

clipped_answer = clip_text(resp_dict["answer"])
print(f"Question:\n{resp_dict['input']}\n\nAnswer:\n{clipped_answer}")
for i, doc in enumerate(resp_dict["context"]):
    print()
    print(f"Source {i + 1}:")
    print(f"  text: {json.dumps(clip_text(doc.page_content))}")
    for key in doc.metadata:
        if key != "pk":
            val = doc.metadata.get(key)
            clipped_val = clip_text(val) if isinstance(val, str) else val
            print(f"  {key}: {clipped_val}")

Question:
Classify this paper into one of the following categories: descriptive, exploratory, inferential, predictive, causal. Also justify why you classified it into that category based on evidences from the paper.

Answer:
The paper can be classified as **exploratory** research. 

Justification: 

1. **Exploration of New Insights**: The paper employs deep learning models and sentiment analysis to explore consumer preferences and satisfaction in the context of online reviews. This indicates an intention to uncover new insights rather than test a specific hypothesis, which is characteristic of exploratory research.

2. **Use of Latent Dirichlet Allocation (LDA)**: The application of LDA for topic extraction and segmentation further supports the exploratory nature of the research. LDA is often used to discover hidden thematic structures in large datasets, which aligns with the goal of exploring and understanding complex consumer sentiments and experiences.

3. **Integration of Business 